# 05 — AI matching, revision 2

**Revision 1 (notebook 04):** best model `paraphrase-multilingual-mpnet-base-v2` → **11/50 on test** vs RapidFuzz **7/50**. +8 points: below our +10 rule → *revise before the deadline*.

**Error analysis of revision 1 (test rows):**
1. **Prepared dishes win**: tomato → *Tomato tart*, salmón → *Salmon tart*, cerdo → *Rillettes, pork*, ceba → *Paëlla*, zanahoria → *Fajitas*.
2. **Right food, wrong state**: chickpea → *cooked* (label: canned), egg → *hard-boiled*, brócoli → *puree*.
3. **Weak on single Spanish/Catalan words**: the model links them to Spanish dishes (paella, fajitas).

**What we change (each option is chosen on TUNE only):**
- `candidates`: remove desserts, sweets, baby food, alcohol, pizzas/tarts, sandwiches, soups (`light`), and also composite dishes (`strict`). ⚠️ `strict` removes *Tofu, plain* (Agribalyse files it as a composite dish): a known cost.
- `passage`: embed the English name **+ the French name** (closer to Spanish/Catalan).
- `model`: add **LaBSE** (trained to align translations) and **bge-m3**.

**Honesty note:** the test split was used once in notebook 04. Here we choose everything on tune and look at test **one more time only**. Both test results go in the log.

**Decision rule (unchanged):** keep AI only if ≥ +10 points vs RapidFuzz on test (≥ 12/50), by the deadline.

> `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all` (~10–15 min, LaBSE and bge-m3 are ~2 GB each).

## 1. Code and data

In [ ]:
# If the repo is private: !git clone https://<YOUR_TOKEN>@github.com/laurarigau/Food-Sustainability.git
!git clone https://github.com/laurarigau/Food-Sustainability.git
%cd Food-Sustainability
!pip install -q -r requirements.txt -r requirements-ai.txt

In [ ]:
import sys, itertools
sys.path.insert(0, "src")
import numpy as np, pandas as pd
from evaluate_matching import build_matcher, evaluate, load_labels
from ai_matching import CANDIDATE_MODELS, EmbeddingMatcher
labels = load_labels()
grid = np.round(np.arange(0, 1.0001, 0.05), 2)

## 2. Baseline (RapidFuzz)

In [ ]:
rf_matcher, rf_grid = build_matcher("rapidfuzz")
rf = evaluate(rf_matcher, rf_grid, labels)
print(f"RapidFuzz TEST: {rf['test_correct']}/{rf['test_n']}")

## 3. Search on TUNE: model × candidates × passage
Only `tune_acc` is used. Test results are computed by the same function but **not shown** in this table.

In [ ]:
rows, runs = [], {}
for model, cand, passage in itertools.product(CANDIDATE_MODELS, ["all", "light", "strict"], ["en", "en_fr"]):
    m = EmbeddingMatcher(model_name=model, threshold=0.0, candidates=cand, passage=passage)
    r = evaluate(m, grid, labels)
    runs[(model, cand, passage)] = r
    rows.append({"model": model.split("/")[-1], "candidates": cand, "passage": passage,
                 "threshold": r["threshold"], "tune_acc": round(r["tune_acc"], 2)})
    print(rows[-1])
tune_table = pd.DataFrame(rows).sort_values("tune_acc", ascending=False)
tune_table.head(10)

In [ ]:
# Choose the configuration with the best TUNE accuracy (ties: first in the sorted table)
best_row = tune_table.iloc[0]
best_key = next(k for k in runs if k[0].split("/")[-1] == best_row.model and k[1] == best_row.candidates and k[2] == best_row.passage)
best = runs[best_key]
print("Chosen on tune:", best_key, "| threshold", best["threshold"], "| tune", f"{best['tune_acc']:.0%}")

## 4. Final result on TEST (second and last look)

In [ ]:
ai_correct, rf_correct, n = best["test_correct"], rf["test_correct"], best["test_n"]
print(f"RapidFuzz          : {rf_correct}/{n} ({rf_correct/n:.0%})")
print(f"AI revision 1 (04) : 11/{n} (22%)")
print(f"AI revision 2      : {ai_correct}/{n} ({ai_correct/n:.0%})  {best_key}")
diff = (ai_correct - rf_correct) / n * 100
print(f"\nDifference vs RapidFuzz: {diff:+.0f} points")
print("DECISION:", "CONTINUE with the AI matcher" if diff >= 10 else "STOP: use RapidFuzz (or labels) and declare it")
pd.DataFrame({
    "rapidfuzz": rf["test"].groupby("language")["correct"].sum(),
    "ai_rev2": best["test"].groupby("language")["correct"].sum(),
    "n": best["test"].groupby("language")["correct"].count(),
})

## 5. Row by row

In [ ]:
cmp = best["test"][["label_id", "ingredient_text", "language", "agribalyse_name"]].copy()
cmp["rapidfuzz_pred"] = rf["test"]["pred_name"].values
cmp["rapidfuzz_ok"] = rf["test"]["correct"].values
cmp["ai_pred"] = np.where(best["test"]["uncertain"], "UNCERTAIN", best["test"]["pred_name"])
cmp["ai_score"] = best["test"]["score"].round(2).values
cmp["ai_ok"] = best["test"]["correct"].values
pd.set_option("display.max_colwidth", 45)
print("AI right, RapidFuzz wrong:", (cmp.ai_ok & ~cmp.rapidfuzz_ok).sum(), "| RapidFuzz right, AI wrong:", (~cmp.ai_ok & cmp.rapidfuzz_ok).sum())
cmp

## 6. Save and download (one zip)

In [ ]:
import shutil
from google.colab import files
best["test"].to_csv("data/results_ai_rev2_test.csv", index=False)
cmp.to_csv("data/comparison_ai_rev2_vs_rapidfuzz_test.csv", index=False)
tune_table.to_csv("data/ai_rev2_tune_search.csv", index=False)
pd.DataFrame([{"model": best_key[0], "candidates": best_key[1], "passage": best_key[2], "threshold": best["threshold"],
               "tune_acc": best["tune_acc"], "test_correct": ai_correct, "rapidfuzz_test_correct": rf_correct, "test_n": n}]
             ).to_csv("data/ai_rev2_summary.csv", index=False)
import os; os.makedirs("/content/out", exist_ok=True)
for f in ["results_ai_rev2_test.csv", "comparison_ai_rev2_vs_rapidfuzz_test.csv", "ai_rev2_tune_search.csv", "ai_rev2_summary.csv"]:
    shutil.copy(f"data/{f}", "/content/out/")
shutil.make_archive("/content/ai_rev2_results", "zip", "/content/out")
files.download("/content/ai_rev2_results.zip")

## 7. Next
- If the decision is CONTINUE: set `DEFAULT_MODEL`, `DEFAULT_THRESHOLD`, `DEFAULT_CANDIDATES`, `DEFAULT_PASSAGE` in `src/ai_matching.py` to the chosen values.
- Save this notebook with outputs (`File → Download → .ipynb`) to `notebooks/05_ai_matching_revision.ipynb`; unzip the CSVs into `data/`.
- Log entry 06: revision 1 and revision 2 numbers, the error analysis, and the decision.